In [2]:
%pip install langchain langchain-community langchain-experimental

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # picks up a .env file if present

DATASET_DIR = Path(r"C:\Users\BIT\Downloads\datasets\datasets")          # adjust if running from elsewhere
MANUALS_DIR = DATASET_DIR / "manuals"
SOP_DIR = DATASET_DIR / "sop"

CHROMA_DIR = Path("./chroma_db")           # persisted vector store lives here
CHROMA_COLLECTION = "plant_documents"

EMBEDDING_MODEL = "all-MiniLM-L6-v2"       # free, local, runs on CPU
GEMINI_MODEL = "gemini-2.5-flash"

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
TOP_K = 5

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")   # set this in your environment or .env
assert GEMINI_API_KEY, "Set GEMINI_API_KEY as an environment variable before running the LLM cells below."

print("Manuals dir exists:", MANUALS_DIR.exists())
print("SOP dir exists:", SOP_DIR.exists())


Manuals dir exists: True
SOP dir exists: True


In [4]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

def load_pdf_documents(root_dir: Path, doc_type: str):
    """Load every PDF under root_dir. Tags each resulting Document with doc_type
    (manual / sop) and category (the immediate parent folder, e.g. 'pumps')."""
    if not root_dir.exists():
        print(f"  (skipping {root_dir}, not found)")
        return []

    loader = DirectoryLoader(
        str(root_dir),
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True,
    )
    docs = loader.load()
    for d in docs:
        source_path = Path(d.metadata.get("source", ""))
        d.metadata["doc_type"] = doc_type
        d.metadata["category"] = source_path.parent.name
        d.metadata["source_file"] = source_path.name
    return docs


manual_docs = load_pdf_documents(MANUALS_DIR, "manual")
sop_docs = load_pdf_documents(SOP_DIR, "sop")

all_docs = manual_docs + sop_docs
print(f"Loaded {len(manual_docs)} manual pages + {len(sop_docs)} SOP pages = {len(all_docs)} total pages")
print("Example metadata:", all_docs[0].metadata if all_docs else None)


C:\Users\BIT\AppData\Local\Temp\ipykernel_25840\2480569963.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
100%|██████████| 16/16 [00:00<00:00, 29.86it/s]

Loaded 817 manual pages + 32 SOP pages = 849 total pages
Example metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2020-03-19T15:45:37-04:00', 'author': 'Doug', 'moddate': '2020-03-19T15:45:37-04:00', 'title': '70006 3_9_20 Boiler.indd', 'source': 'C:\\Users\\BIT\\Downloads\\datasets\\datasets\\manuals\\boilers\\boiler_oem_manual.pdf', 'total_pages': 54, 'page': 0, 'page_label': '1', 'doc_type': 'manual', 'category': 'boilers', 'source_file': 'boiler_oem_manual.pdf'}


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(all_docs)
print(f"{len(all_docs)} pages -> {len(chunks)} chunks")
print("Example chunk:\n", chunks[0].page_content[:300])
print("Metadata:", chunks[0].metadata)


849 pages -> 2656 chunks
Example chunk:
 OWNER’S MANUAL
With CE-4 Burner
Clean Energy Heating Systems, LLC
PO Box 1341, 540 Maple Street, Honey Brook, PA 19344
www.CleanEnergyHeatingSystems.com
(888) 519-2347
vapors and liquids in the vicinity of this or any appliance.
H

Metadata: {'producer': 'Microsoft: Print To PDF', 'creator': 'PyPDF', 'creationdate': '2020-03-19T15:45:37-04:00', 'author': 'Doug', 'moddate': '2020-03-19T15:45:37-04:00', 'title': '70006 3_9_20 Boiler.indd', 'source': 'C:\\Users\\BIT\\Downloads\\datasets\\datasets\\manuals\\boilers\\boiler_oem_manual.pdf', 'total_pages': 54, 'page': 0, 'page_label': '1', 'doc_type': 'manual', 'category': 'boilers', 'source_file': 'boiler_oem_manual.pdf'}


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = Chroma(
    collection_name=CHROMA_COLLECTION,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR),
)

# Add in batches so this is memory-friendly on the 355-page control valve manual etc.
BATCH_SIZE = 64
for i in range(0, len(chunks), BATCH_SIZE):
    batch = chunks[i:i + BATCH_SIZE]
    vectorstore.add_documents(batch)
    print(f"  ...indexed {min(i + BATCH_SIZE, len(chunks))}/{len(chunks)} chunks")

print("Vector store built and persisted at", CHROMA_DIR.resolve())


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  ...indexed 64/2656 chunks
  ...indexed 128/2656 chunks
  ...indexed 192/2656 chunks
  ...indexed 256/2656 chunks
  ...indexed 320/2656 chunks
  ...indexed 384/2656 chunks
  ...indexed 448/2656 chunks
  ...indexed 512/2656 chunks
  ...indexed 576/2656 chunks
  ...indexed 640/2656 chunks
  ...indexed 704/2656 chunks
  ...indexed 768/2656 chunks
  ...indexed 832/2656 chunks
  ...indexed 896/2656 chunks
  ...indexed 960/2656 chunks
  ...indexed 1024/2656 chunks
  ...indexed 1088/2656 chunks
  ...indexed 1152/2656 chunks
  ...indexed 1216/2656 chunks
  ...indexed 1280/2656 chunks
  ...indexed 1344/2656 chunks
  ...indexed 1408/2656 chunks
  ...indexed 1472/2656 chunks
  ...indexed 1536/2656 chunks
  ...indexed 1600/2656 chunks
  ...indexed 1664/2656 chunks
  ...indexed 1728/2656 chunks
  ...indexed 1792/2656 chunks
  ...indexed 1856/2656 chunks
  ...indexed 1920/2656 chunks
  ...indexed 1984/2656 chunks
  ...indexed 2048/2656 chunks
  ...indexed 2112/2656 chunks
  ...indexed 2176/2656 chu

In [7]:
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})

# quick sanity check
test_hits = retriever.invoke("boiler startup procedure")
for h in test_hits:
    print(f"- {h.metadata.get('source_file')} (page {h.metadata.get('page')}): {h.page_content[:120]}...")


- sop_boiler_startup_sop.pdf (page 0): Boiler Startup Procedure
Document ID SOP003 Category Utility
Revision 3.0 Effective Date 2024-02-18
1. Purpose
To initia...
- sop_boiler_startup_sop.pdf (page 0): Boiler Startup Procedure
Document ID SOP003 Category Utility
Revision 3.0 Effective Date 2024-02-18
1. Purpose
To initia...
- sop_boiler_startup_sop.pdf (page 0): Boiler Startup Procedure
Document ID SOP003 Category Utility
Revision 3.0 Effective Date 2024-02-18
1. Purpose
To initia...
- sop_boiler_startup_sop.pdf (page 0): Boiler Startup Procedure
Document ID SOP003 Category Utility
Revision 3.0 Effective Date 2024-02-18
1. Purpose
To initia...
- sop_boiler_startup_sop.pdf (page 0): Boiler Startup Procedure
Document ID SOP003 Category Utility
Revision 3.0 Effective Date 2024-02-18
1. Purpose
To initia...


In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=GEMINI_API_KEY,
    
)

SYSTEM_PROMPT = """You are a plant maintenance and safety assistant.
Answer ONLY using the context below, which comes from OEM manuals and SOPs.
Always cite the source file and page for any fact you use, like: (source_file, p.X).
If the context does not contain the answer, say so plainly instead of guessing.
Keep the answer concise and practical for a maintenance engineer."""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "Context:\n{context}\n\nQuestion: {question}"),
])


def format_docs(docs):
    parts = []
    for d in docs:
        src = d.metadata.get("source_file", "unknown")
        page = d.metadata.get("page", "?")
        parts.append(f"[{src}, p.{page}] {d.page_content}")
    return "\n\n".join(parts)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)


In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI

test_llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    google_api_key=GEMINI_API_KEY
)

response = test_llm.invoke("Say hello")
print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


ChatGoogleGenerativeAIError: Error calling model 'gemini-3.5-flash' (PERMISSION_DENIED): 403 PERMISSION_DENIED. {'error': {'code': 403, 'message': 'Your project has been denied access. Please contact support.', 'status': 'PERMISSION_DENIED'}}

In [9]:
def ask(question: str):
    print(f"Q: {question}\n")
    answer = rag_chain.invoke(question)
    print("A:", answer)
    print("\n--- Sources retrieved ---")
    for d in retriever.invoke(question):
        print(f"- {d.metadata.get('source_file')} (page {d.metadata.get('page')})")
    print("=" * 80)


ask("What is the startup procedure for the boiler?")


Q: What is the startup procedure for the boiler?

A: The step-by-step startup procedure for the B-101 and B-102 Steam Boilers is as follows:

1. **Verify water level:** Confirm that the boiler water level is at the Normal Water Level (NWL) in the sight glass.
2. **Open fuel supply:** Open the main gas supply valve.
3. **Pre-purge:** Start the forced draft (FD) fan and initiate the 5-minute pre-purge cycle.
4. **Ignite pilot:** Verify that the pilot ignition sequence starts.
5. **Open burner valves:** Once the pilot is confirmed, open the main burner gas valves.
6. **Monitor flame:** Monitor the flame stability through the view port.
7. **Close vents:** Close the atmospheric vents once the steam pressure reaches 2 bar.

**(source_file: sop_boiler_startup_sop.pdf, p.0)**

--- Sources retrieved ---
- sop_boiler_startup_sop.pdf (page 0)
- sop_boiler_startup_sop.pdf (page 0)
- sop_boiler_startup_sop.pdf (page 0)
- sop_boiler_startup_sop.pdf (page 0)
- sop_boiler_shutdown_sop.pdf (page 0)


In [10]:
def ask(question: str):
    print(f"Q: {question}\n")
    answer = rag_chain.invoke(question)
    print("A:", answer)
    print("\n--- Sources retrieved ---")
    for d in retriever.invoke(question):
        print(f"- {d.metadata.get('source_file')} (page {d.metadata.get('page')})")
    print("=" * 80)

ask("What PPE is required before working on the control valve?")

Q: What PPE is required before working on the control valve?

A: The provided context does not contain information about the PPE required before working on the control valve.

--- Sources retrieved ---
- control_valve_oem_manual.pdf (page 0)
- control_valve_oem_manual.pdf (page 0)
- control_valve_oem_manual.pdf (page 0)
- control_valve_oem_manual.pdf (page 0)
- control_valve_oem_manual.pdf (page 338)


In [11]:
def ask(question: str):
    print(f"Q: {question}\n")
    answer = rag_chain.invoke(question)
    print("A:", answer)
    print("\n--- Sources retrieved ---")
    for d in retriever.invoke(question):
        print(f"- {d.metadata.get('source_file')} (page {d.metadata.get('page')})")
    print("=" * 80)

ask("What PPE is required before working on the control valve?")

Q: What PPE is required before working on the control valve?

A: Based on the provided context, there is no information regarding the PPE required before working on the control valve.

--- Sources retrieved ---
- control_valve_oem_manual.pdf (page 0)
- control_valve_oem_manual.pdf (page 0)
- control_valve_oem_manual.pdf (page 0)
- control_valve_oem_manual.pdf (page 0)
- control_valve_oem_manual.pdf (page 338)
